# MUSE vs ProtoECGNet: bootstrapped balanced accuracy

Bootstrapped 95% CIs for balanced accuracy of GE MUSE vs the branched and fusion ProtoECGNet models on the HEEDB by-label test set, using the physician overread as ground truth (`label_src`: `original_physician` or `audited_physician`).

- Model thresholds are picked on val (Youden's J, as in `muse-accuracy.ipynb`) and held fixed; only the test set is resampled. MUSE only has hard labels, so only balanced accuracy is compared.
- **Joint patient-level bootstrap**: each replicate resamples test patients with replacement, keeping all of a patient's ECGs (they are correlated), and the same resample is used for every label and every model. This gives a CI on the macro average and paired CIs on model differences.
- Replicates are not filtered or stratified on label prevalence: the test set is a simple random sample of patients, so prevalence variation is part of the sampling variability.
- CIs are percentile intervals of the bootstrap distribution.
- With `audited_physician`, some (ECG, label) entries are unknown (`UNKNOWN_LABEL`, see `protossl/datasets/_heedb_label_rules.py`). Each label's threshold, point estimate and bootstrap counts use only the ECGs where that label is known. MUSE predictions then come from `audited_muse`, which only counts codes in MUSE's main read.

In [1]:
from protossl.datasets import HeedbECGDataset
from protossl.defines import HEEDB_TARGETS, UNKNOWN_LABEL
from omegaconf import OmegaConf
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, balanced_accuracy_score
from tqdm.auto import tqdm

dataset_path = "/opt/gpudata/ecg/heedb"
arms = ["2D-partial", "2D-global", "1D-global"]
run_path = Path("/opt/gpu_working/steven/protoecgnet-heedb-v4")
config_path = Path("/opt/gpu_working/steven/ProtoSSL/large-user-study/configs")
fusion_path = run_path / "fusion/train-fusion-classifier/latest"
# ground truth labels: "original_physician" or "audited_physician" (has unknowns)
label_src = "audited_physician"
# MUSE predictions read the same way as the ground truth: audited_muse only counts main-read
# codes, original_muse also counts the comparison text (which often describes a prior ECG)
muse_label_src = "original_muse" if label_src == "original_physician" else "audited_muse"
# original_physician keeps the original file name
csv_suffix = "" if label_src == "original_physician" else f"-{label_src}"
output_csv = run_path / f"muse-comparison-bootstrapped{csv_suffix}.csv"
fusion_labels = list(HEEDB_TARGETS)
label_idxs = np.arange(len(HEEDB_TARGETS))

n_bootstraps = 1000
seed = 0
models = ["muse", "branched", "fusion"]
comparisons = [("fusion", "muse"), ("branched", "muse")]

In [2]:
def stack_branched_arms():
    arm_labels = [l for arm in arms for l in OmegaConf.load(config_path / f"{arm}.yaml").data.init_args.label_subset]
    idxs = np.asarray([arm_labels.index(l) for l in fusion_labels]) # indices to normalize to fusion order

    ret = dict()
    for split in ["val", "test"] :
        arm_probs = {arm: np.load(run_path / arm / f"train-classifier/latest/{split}_probs.npy") for arm in arms}
        stacked_probs = np.concat([arm_probs[arm] for arm in arms], axis=1)
        ret[split] = stacked_probs[:, idxs]
    return ret

In [3]:
ds_phys = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src=label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

ds_phys_val = HeedbECGDataset(
    dataset_path=dataset_path,
    split="val",
    sampling_rate=100,
    label_src=label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

ds_muse = HeedbECGDataset(
    dataset_path=dataset_path,
    split="test",
    sampling_rate=100,
    label_src=muse_label_src,
    heedb_split_type="by-label",
    drop_blank_overread=True,
)

================get_heedb_metadata=================
using by-label splits
reading HEEDB metadata from on-disk cache: /home/songs1/.cache/protossl_cache/abac20ad.csv
==============get_heedb_overread_mask==============
reading HEEDB overread mask from on-disk cache: /home/songs1/.cache/protossl_cache/22bb7f8c.npy
replaying below stats from cache:
Of 455748 ECGs, keeping 362090 with a physician overread, dropping 93658 with a blank overread and 0 not found in the diagnoses table
=================make_heedb_labels=================
reading HEEDB labels from on-disk cache: /home/songs1/.cache/protossl_cache/02945b1e.npy
replaying below stats from cache:
Of 362090 ECGs and 11607259 annotations, 362090 matched (0 ECG annotations were missing and filled with 0s)
===============StreamingECGWaveforms===============
Using streaming ECG waveforms, will load and transform data on the fly
==============get_heedb_overread_mask==============
reading HEEDB overread mask from on-disk cache: /home/songs1/

In [4]:
y_true = ds_phys.labels.numpy()[:, label_idxs]
y_true_val = ds_phys_val.labels.numpy()[:, label_idxs]
y_pred_muse = ds_muse.labels.numpy()[:, label_idxs]
# per-label masks of ECGs with a known ground truth label
known = y_true != UNKNOWN_LABEL
known_val = y_true_val != UNKNOWN_LABEL
assert (y_pred_muse != UNKNOWN_LABEL).all()

patient_ids = ds_phys.source_ids.numpy()
assert (patient_ids == ds_muse.source_ids.numpy()).all()
assert (ds_phys.sample_ids.numpy() == ds_muse.sample_ids.numpy()).all()
# maps each ECG to its patient's position in [0, n_patients)
_, patient_idx = np.unique(patient_ids, return_inverse=True)
n_patients = patient_idx.max() + 1
print(f"{len(patient_ids)} test ECGs from {n_patients} patients")
for i in np.flatnonzero(~known.all(axis=0)):
    print(f"{fusion_labels[i]}: {(~known[:, i]).sum()} test / {(~known_val[:, i]).sum()} val unknown")

362090 test ECGs from 77596 patients


In [5]:
def compute_threshold(_y_true, _y_prob):
    # NOTE: should be computed on either train/val set, NOT test
    fpr, tpr, thresholds = roc_curve(_y_true, _y_prob)
    j_scores = tpr - fpr
    optimal_idx = np.argmax(j_scores)
    optimal_threshold = thresholds[optimal_idx]
    return optimal_threshold

In [6]:
temp = stack_branched_arms()
y_prob_branched, y_prob_branched_val = temp["test"], temp["val"]
y_prob_fusion = np.load(fusion_path / "test_probs.npy")
y_prob_fusion_val = np.load(fusion_path / "val_probs.npy")
assert y_prob_fusion.shape[1] == len(fusion_labels)

In [7]:
y_pred_branched = np.zeros_like(y_pred_muse)
y_pred_fusion = np.zeros_like(y_pred_muse)
for i, label in enumerate(fusion_labels):
    kv = known_val[:, i]
    branched_threshold = compute_threshold(y_true_val[kv, i], y_prob_branched_val[kv, i])
    y_pred_branched[:, i] = (y_prob_branched[:, i] > branched_threshold).astype(int)
    fusion_threshold = compute_threshold(y_true_val[kv, i], y_prob_fusion_val[kv, i])
    y_pred_fusion[:, i] = (y_prob_fusion[:, i] > fusion_threshold).astype(int)

y_pred = {"muse": y_pred_muse, "branched": y_pred_branched, "fusion": y_pred_fusion}
point = {
    m: np.asarray([
        balanced_accuracy_score(y_true[known[:, i], i], y_pred[m][known[:, i], i])
        for i in range(len(fusion_labels))
    ])
    for m in models
}

## Joint patient-level bootstrap

A resample is represented by per-ECG weights `w` (how many times each ECG's patient was drawn), so every confusion-matrix count over the resampled set is a weighted sum, e.g. `TP = w @ (y_true & y_pred)`.

In [8]:
# unknown labels are in neither pos nor neg, so they drop out of every count
pos, neg = y_true == 1, y_true == 0
pos_f, neg_f = pos.astype(np.float64), neg.astype(np.float64)
tp_ind = {m: (pos & (y_pred[m] == 1)).astype(np.float64) for m in models}
tn_ind = {m: (neg & (y_pred[m] == 0)).astype(np.float64) for m in models}


def weighted_balacc(w, m):
    sens = (w @ tp_ind[m]) / (w @ pos_f)
    spec = (w @ tn_ind[m]) / (w @ neg_f)
    return (sens + spec) / 2


# with unit weights this must reproduce the point estimates
for m in models:
    assert np.allclose(weighted_balacc(np.ones(len(y_true)), m), point[m])

In [9]:
rng = np.random.default_rng(seed)
boot = {m: np.empty((n_bootstraps, len(fusion_labels))) for m in models}
for b in tqdm(range(n_bootstraps)):
    # resample patients, then weight each ECG by how many times its patient was drawn
    counts = np.bincount(rng.integers(0, n_patients, n_patients), minlength=n_patients)
    w = counts[patient_idx].astype(np.float64)
    for m in models:
        boot[m][b] = weighted_balacc(w, m)

  0%|          | 0/1000 [00:00<?, ?it/s]

In [10]:
def ci(a):
    return np.percentile(a, [2.5, 97.5], axis=0)


row_names = fusion_labels + ["Macro Average"]
# append the macro average as a final column, computed per replicate
est = {m: np.append(point[m], point[m].mean()) for m in models}
dist = {m: np.column_stack([boot[m], boot[m].mean(axis=1)]) for m in models}

results = pd.DataFrame(index=pd.Index(row_names, name="label"))
for m in models:
    lo, hi = ci(dist[m])
    results[f"{m}: BalAcc"] = est[m]
    results[f"{m}: 95% CI (lo)"] = lo
    results[f"{m}: 95% CI (hi)"] = hi
for a, b in comparisons:
    lo, hi = ci(dist[a] - dist[b])
    results[f"{a} - {b}: diff"] = est[a] - est[b]
    results[f"{a} - {b}: 95% CI (lo)"] = lo
    results[f"{a} - {b}: 95% CI (hi)"] = hi
    results[f"{a} vs {b}: CIs overlap"] = (
        (results[f"{a}: 95% CI (lo)"] <= results[f"{b}: 95% CI (hi)"])
        & (results[f"{b}: 95% CI (lo)"] <= results[f"{a}: 95% CI (hi)"])
    )
    results[f"{a} vs {b}: diff CI excludes 0"] = (lo > 0) | (hi < 0)

output_csv.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(output_csv)
print(f"Saved results to {output_csv}")

Saved results to /opt/gpu_working/steven/protoecgnet-heedb-v3/muse-comparison-bootstrapped.csv


In [11]:
def fmt(e, lo, hi, signed=False):
    s = "+.3f" if signed else ".3f"
    return f"{e:{s}} ({lo:{s}}, {hi:{s}})"


model_table = pd.DataFrame(
    {
        m: [fmt(*r) for r in results[[f"{m}: BalAcc", f"{m}: 95% CI (lo)", f"{m}: 95% CI (hi)"]].values]
        for m in models
    },
    index=results.index,
)
display(model_table[["muse", "fusion"]])

,muse,fusion
label,,
ANTERIOR INFARCT,"0.950 (0.945, 0.955)","0.890 (0.885, 0.895)"
ATRIAL FIBRILLATION,"0.933 (0.930, 0.936)","0.918 (0.915, 0.920)"
ATRIAL FLUTTER,"0.851 (0.843, 0.860)","0.894 (0.890, 0.899)"
ATRIAL-PACED RHYTHM,"0.846 (0.829, 0.863)","0.944 (0.936, 0.951)"
INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,"0.907 (0.902, 0.912)","0.911 (0.907, 0.914)"
INFERIOR INFARCT,"0.975 (0.972, 0.978)","0.912 (0.907, 0.917)"
LATERAL INFARCT,"0.973 (0.970, 0.977)","0.907 (0.903, 0.912)"
LEFT BUNDLE BRANCH BLOCK,"0.939 (0.933, 0.946)","0.958 (0.955, 0.961)"
NORMAL SINUS RHYTHM,"0.938 (0.936, 0.939)","0.910 (0.909, 0.912)"


In [12]:
comparison_table = pd.DataFrame(index=results.index)
for a, b in comparisons:
    cols = [f"{a} - {b}: diff", f"{a} - {b}: 95% CI (lo)", f"{a} - {b}: 95% CI (hi)"]
    comparison_table[f"{a} - {b}"] = [fmt(*r, signed=True) for r in results[cols].values]
    comparison_table[f"{a} vs {b}: CIs overlap"] = results[f"{a} vs {b}: CIs overlap"]
    comparison_table[f"{a} vs {b}: diff CI excludes 0"] = results[f"{a} vs {b}: diff CI excludes 0"]
display(comparison_table)

,fusion - muse,fusion vs muse: CIs overlap,fusion vs muse: diff CI excludes 0,branched - muse,branched vs muse: CIs overlap,branched vs muse: diff CI excludes 0
label,,,,,,
ANTERIOR INFARCT,"-0.060 (-0.066, -0.055)",False,True,"-0.059 (-0.065, -0.053)",False,True
ATRIAL FIBRILLATION,"-0.015 (-0.018, -0.012)",False,True,"-0.015 (-0.018, -0.012)",False,True
ATRIAL FLUTTER,"+0.043 (+0.033, +0.052)",False,True,"+0.044 (+0.034, +0.053)",False,True
ATRIAL-PACED RHYTHM,"+0.098 (+0.082, +0.114)",False,True,"+0.096 (+0.081, +0.112)",False,True
INCOMPLETE RIGHT BUNDLE BRANCH BLOCK,"+0.003 (-0.002, +0.008)",True,False,"+0.004 (-0.001, +0.009)",True,False
INFERIOR INFARCT,"-0.063 (-0.069, -0.058)",False,True,"-0.063 (-0.069, -0.058)",False,True
LATERAL INFARCT,"-0.066 (-0.071, -0.062)",False,True,"-0.065 (-0.070, -0.060)",False,True
LEFT BUNDLE BRANCH BLOCK,"+0.019 (+0.013, +0.025)",False,True,"+0.019 (+0.013, +0.025)",False,True
NORMAL SINUS RHYTHM,"-0.027 (-0.028, -0.027)",False,True,"-0.027 (-0.028, -0.027)",False,True
